# Séance 3 — Nettoyage des données

**Analyse des données — L3 Économie**

Deux fichiers aujourd'hui.

| Fichier | Ce qu'il contient | Où il sert |
|---|---|---|
| `cm02-communes.csv` | 45 communes, une ligne par commune | reprise, et premiers manquants |
| `cm03-salaires.csv` | 8 012 postes salariés, une ligne par poste | tout le reste |

**Ouvrez les deux dictionnaires avant de commencer.** Ils sont sur le site du cours. Sans eux, la moitié de ce notebook n'a pas de sens : le dictionnaire est la seule source qui dise comment la non-réponse est codée.

> Ces données sont fabriquées pour l'enseignement. Les ordres de grandeur sont plausibles, les valeurs ne sont pas celles de l'INSEE.

*Exécution → Tout exécuter* avant de démarrer.

## 1. Les bibliothèques et les fichiers

In [ ]:
import pandas as pd
import numpy as np

BASE = ("https://raw.githubusercontent.com/StefaniaMarcassa/"
        "analyse_des_donnees/main/data/")
# Pour travailler en local, remplacer par : BASE = ""

In [ ]:
com = pd.read_csv(
    BASE + "cm02-communes.csv",
    encoding="latin-1", sep=";", decimal=",", thousands=" ",
    dtype={"CODGEO": str, "CAT_URBAINE": str}
)

# Les deux conversions faites en seance 2
com["TAUX_CHOMAGE"] = pd.to_numeric(
    com["TAUX_CHOMAGE"].astype(str).str.replace(",", ".", regex=False),
    errors="coerce"
)
com["NB_ENTREPRISES"] = com["NB_ENTREPRISES"].replace(9999, np.nan)

print(com.shape)
com.head(3)

---

## 2. Les valeurs manquantes : les trouver

Deux commandes, toujours les mêmes.

In [ ]:
print(com.isna().sum())
print()
print(com.isna().mean().round(3))

Deux colonnes concernées : `TAUX_CHOMAGE` (6 communes, 13,3 %) et `NB_ENTREPRISES` (2 communes, 4,4 %).

Ces manquants n'existaient pas au chargement. Ils sont le résultat de nos conversions : le `nd` et le `9999` étaient invisibles pour `isna()`.

## 3. La question qui compte : *qui* sont les manquants ?

Compter les manquants ne coûte rien et n'apprend rien. La question utile est de savoir **sur quelles lignes** ils tombent.

In [ ]:
com.groupby("CAT_URBAINE")["TAUX_CHOMAGE"].apply(lambda s: s.isna().sum())

In [ ]:
# La meme chose en part, avec l'effectif de chaque categorie
resume = pd.DataFrame({
    "communes": com["CAT_URBAINE"].value_counts().sort_index(),
    "sans_taux": com.groupby("CAT_URBAINE")["TAUX_CHOMAGE"].apply(lambda s: s.isna().sum()),
})
resume["part"] = (resume["sans_taux"] / resume["communes"]).round(2)
resume

**Les six valeurs absentes sont les six communes rurales du fichier, et elles seulement.**

L'absence n'est pas un accident. Elle est produite par le secret statistique, qui dépend de la taille de la commune. Une variable que vous observez — `CAT_URBAINE` — explique parfaitement ce qui manque.

C'est le deuxième des trois cas vus en cours : **manquant au hasard, conditionnellement**. On peut en tenir compte, à condition de l'avoir vu.

In [ ]:
# Ce que coute la suppression
propre = com.dropna(subset=["TAUX_CHOMAGE", "NB_ENTREPRISES"])
print(len(com), "->", len(propre))

print("population moyenne, conservees :", round(com.dropna(subset=["TAUX_CHOMAGE"])["POP"].mean()))
print("population moyenne, supprimees :", round(com[com["TAUX_CHOMAGE"].isna()]["POP"].mean()))

Huit lignes sur quarante-cinq. Le nombre n'est pas le problème : la population moyenne passe de 156 380 à 1 786 entre les deux groupes.

**Le fichier nettoyé ne décrit plus les communes françaises.** Il décrit les communes de plus de trois mille habitants. C'est peut-être ce que vous vouliez ; il faut alors l'écrire.

---

## 4. Un second fichier

Quarante-cinq communes ne suffisent pas pour la suite. Nous passons à des données individuelles.

In [ ]:
brut = pd.read_csv(
    BASE + "cm03-salaires.csv",
    encoding="latin-1", sep=";", decimal=",", thousands=" ",
    dtype={"IDENT": str, "SECTEUR": str, "REGION": str}
)
df = brut.copy()          # on ne touche jamais a `brut`

print(brut.shape)
brut.head()

**8 012 lignes, 15 colonnes.**

Une ligne est **un poste occupé dans l'année**, pas une personne. Retenez-le : nous y reviendrons à la section 10, et c'est de là que viendront les doublons.

Notez aussi `IDENT`, chargé en texte. Lu comme un entier il perdrait ses zéros initiaux, exactement comme `CODGEO` en séance 2.

## 5. Le calcul qui fonctionne, et qui est faux

In [ ]:
df["SALAIRE_NET"].mean()

Soixante-sept mille euros de salaire mensuel net. Aucune erreur, aucun avertissement, un résultat absurde.

Au lieu de moyenner la colonne, regardons-la.

In [ ]:
df["SALAIRE_NET"].value_counts().head(4)

Trois valeurs reviennent des centaines de fois. Le dictionnaire dit ce que chacune signifie, et les trois ne veulent pas dire la même chose.

| Valeur | Effectif | Ce que dit le dictionnaire | Ce qu'il faut en faire |
|---|---|---|---|
| `0` | 675 | Sans objet : non-salariés | exclure du champ |
| `999999` | 525 | Non-réponse | passer en manquant |
| `10000` | 72 | Plafond de diffusion | garder, et le signaler |
| `1859.27` | 3 | Rien | rien |

**Trois problèmes différents dans une seule colonne.** Aucun des trois ne se traite comme les autres.

**Et une quatrième ligne qui n'en est pas un.** Trois salariés au même euro près, c'est une coïncidence, pas un code. Le critère n'est donc pas la répétition seule : c'est l'écart d'ordre de grandeur.

In [ ]:
# Verifier que la repetition est bien anormale, plutot que de le supposer
effectifs = df["SALAIRE_NET"].value_counts().drop([0, 999999, 10000])

print("valeurs distinctes hors codes :", len(effectifs))
print("part vue une seule fois       :", round(100 * (effectifs == 1).mean(), 1), "%")
print("effectif maximal              :", effectifs.max())

Sur 8 012 lignes, 98,9 % des salaires n'apparaissent qu'une fois et aucun ne revient plus de trois fois. Trois codes à 675, 525 et 72 sortent de plusieurs ordres de grandeur.

Sur le fichier des communes, quarante-cinq lignes, une seule répétition suffisait à alerter. La règle dépend de la taille du fichier ; le réflexe de regarder, non.

In [ ]:
# 1. La non-reponse devient un manquant
df["SALAIRE_NET"] = df["SALAIRE_NET"].replace(999999, np.nan)
print("apres conversion :", round(df["SALAIRE_NET"].mean(), 2),
      "sur", df["SALAIRE_NET"].notna().sum(), "observations")

In [ ]:
# 2. Les non-salaries sortent du champ : ils n'ont pas un salaire nul,
#    ils n'ont pas de salaire.
champ = df.loc[df["STATUT"] != 3]
print("champ salarie :", len(champ), "lignes")
print("salaire moyen :", round(champ["SALAIRE_NET"].mean(), 2),
      "sur", champ["SALAIRE_NET"].notna().sum(), "observations")

De 67 745 à 2 375, puis à 2 610. Une ligne de code pour la première correction, un facteur vingt-huit.

**Aucune erreur ne s'affiche jamais pour ce genre de problème.** Seul le dictionnaire vous prévient.

## 6. Qui sont les manquants, ici ?

In [ ]:
diag = brut.copy()
diag["nr_salaire"] = diag["SALAIRE_NET"] == 999999

(100 * diag.groupby("DIPLOME")["nr_salaire"].mean()).round(1)

Le taux de non-réponse passe de 3,3 % chez les non-diplômés à 10,6 % chez les bac+5. Il est trois fois plus élevé en haut de l'échelle qu'en bas.

**La huitième ligne n'est pas un niveau de diplôme.** La modalité 9 est la non-réponse au diplôme. Ses 5,9 % ne se lisent pas sur le gradient, puisque ce groupe mélange tous les niveaux. Une modalité « non-réponse » qui apparaît dans un profil est une erreur de nettoyage, pas un résultat : sur un vrai tableau, `DIPLOME` se convertit avant d'être croisé avec quoi que ce soit.

Ce tableau **ne prouve pas** que la non-réponse dépend du salaire lui-même. Il montre qu'elle dépend d'une variable qui lui est fortement liée, ce qui rend l'hypothèse de hasard intenable.

C'est le troisième cas : **manquant non au hasard**. Aucune correction n'est possible à partir des seules données observées, par construction — on ne voit pas ce qui manque. Ce qu'on peut faire, c'est le dire, et mesurer la sensibilité du résultat.

## 7. Hors champ n'est pas non-réponse

Trois colonnes contiennent des cases vides. Aucune des trois ne signifie la même chose.

In [ ]:
print("SALAIRE_NET a 0    :", (brut["SALAIRE_NET"] == 0).sum(), " -> non-salaries, sans objet")
print("MOTIF_ABS vide     :", brut["MOTIF_ABS"].isna().sum(), " -> n'a pas ete absent, sans objet")
print("PRIME vide         :", brut["PRIME"].isna().sum(), " -> n'a touche aucune prime")

Une personne **hors champ** n'aurait pas dû répondre. Une **non-réponse** aurait dû. Les traiter de la même façon est une faute.

Remplacer une prime absente par la prime moyenne invente un versement qui n'a pas eu lieu. Supprimer la ligne élimine précisément les salariés sans prime, c'est-à-dire la majorité d'entre eux.

## 8. Le piège : la modalité 9

Regardons `MOTIF_ABS`, le motif d'absence de la semaine de référence.

In [ ]:
df["MOTIF_ABS"].value_counts().sort_index()

Le dictionnaire donne les libellés :

| Code | Libellé | Effectif |
|---|---|---|
| 1 | Congés | 458 |
| 2 | Maladie | 175 |
| 3 | Congé de maternité ou de paternité | 34 |
| 4 | Formation | 54 |
| 5 | Raisons familiales | 45 |
| **9** | **Chômage partiel** | **61** |
| 99 | Non-réponse | 32 |

**Ici la non-réponse est codée 99, et 9 est une vraie situation.**

In [ ]:
# Le geste qu'il ne faut PAS faire
essai = df["MOTIF_ABS"].replace(9, np.nan)
print("chomage partiel avant :", (df["MOTIF_ABS"] == 9).sum())
print("chomage partiel apres :", (essai == 9).sum())

Soixante et une situations de chômage partiel effacées, sans message d'erreur, et c'est exactement la catégorie qu'un économiste du travail cherchait.

Trois colonnes de ce fichier utilisent le code 9, et il ne veut pas dire la même chose dans les trois. **Lisez la ligne de la variable, pas la colonne du code.**

## 9. La censure haute

Une valeur qui revient beaucoup trop souvent dans une variable continue signale un plafond.

In [ ]:
df["HEURES"].value_counts().sort_index().tail(8)

Deux cent soixante-treize personnes à exactement 60 heures, contre une trentaine à 58 ou 59. Le dictionnaire tranche : la modalité se lit **« 60 heures ou plus »**.

Ce n'est pas une erreur. C'est un regroupement décidé par le producteur, et la moyenne des heures est donc une **borne inférieure**.

In [ ]:
df["HEURES"] = df["HEURES"].replace(99, np.nan)      # 99 = non-reponse
print("duree moyenne declaree :", round(df["HEURES"].mean(), 2), "heures")
print("part a 60 heures ou plus :", round(100 * (df["HEURES"] == 60).mean(), 2), "%")

In [ ]:
# Meme chose sur le salaire : les quantiles hauts se rejoignent
champ = df.loc[df["STATUT"] != 3]
champ["SALAIRE_NET"].quantile([0.50, 0.90, 0.95, 0.99, 0.999]).round(2)

Les quantiles à 99 % et à 99,9 % valent tous deux 10 000 euros : la distribution est tronquée à son sommet.

**Un indice de Gini calculé sur cette colonne est une borne inférieure.** La censure comprime le haut de la distribution, là où se concentre l'essentiel de l'inégalité mesurée.

La phrase à écrire dans un mémoire est de cette forme : *« les rémunérations supérieures au plafond de diffusion sont ramenées à ce plafond ; les indicateurs d'inégalité présentés constituent donc des bornes inférieures »*.

## 10. Les doublons : deux questions différentes

In [ ]:
print("lignes identiques partout :", brut.duplicated().sum())
print("IDENT repete              :", brut.duplicated(subset=["IDENT"]).sum())
print("IDENT + NOPOSTE repete    :", brut.duplicated(subset=["IDENT", "NOPOSTE"]).sum())

Douze lignes strictement identiques : import répété, à supprimer.

Six cent trente-deux identifiants répétés : ce n'est pas la même chose. Une ligne est un **poste**, et une personne peut en occuper deux dans l'année.

**La clé du fichier n'est pas `IDENT`, c'est le couple `IDENT` + `NOPOSTE`.**

In [ ]:
doublons = brut[brut.duplicated(subset=["IDENT"], keep=False)]
(doublons
 .sort_values(["IDENT", "NOPOSTE"])
 [["IDENT", "NOPOSTE", "STATUT", "TEMPS", "HEURES", "SALAIRE_NET", "SECTEUR"]]
 .head(6))

Les deux lignes d'une même personne diffèrent sur le temps de travail, les heures, le salaire et le secteur. Ce sont deux emplois distincts.

L'ordre des opérations compte : on retire d'abord les doublons techniques, et on garde ce qui reste.

In [ ]:
df = brut.drop_duplicates().copy()
print(len(df), "lignes,", df["IDENT"].nunique(), "personnes")
print("identifiants encore repetes :", df.duplicated(subset=["IDENT"]).sum())

---

## 11. Écrire votre première fonction

Quatre colonnes codent la non-réponse par un nombre. Une règle appliquée quatre fois s'écrit une fois.

In [ ]:
def nettoyer(serie, codes_manquants):
    """Convertit en numerique, puis met les codes de non-reponse en manquant."""
    s = pd.to_numeric(serie, errors="coerce")
    return s.replace(codes_manquants, np.nan)


df["SALAIRE_NET"] = nettoyer(df["SALAIRE_NET"], [999999])
df["PRIME"]       = nettoyer(df["PRIME"],       [999999])
df["HEURES"]      = nettoyer(df["HEURES"],      [99])
df["DIPLOME"]     = nettoyer(df["DIPLOME"],     [9])

df[["SALAIRE_NET", "PRIME", "HEURES", "DIPLOME"]].isna().sum()

Trois bénéfices : la règle est écrite une seule fois, elle porte un nom qui dit ce qu'elle fait, et la corriger la corrige partout.

**Notez la ligne absente : `MOTIF_ABS`.** La fonction est correcte ; la liste de codes ne l'est jamais par défaut. Chaque colonne a la sienne, et elle vient du dictionnaire.

## 12. Décider : les trois options

| Option | Ce qu'on fait | Ce que cela coûte |
|---|---|---|
| Supprimer les lignes | on retire les observations incomplètes | échantillon réduit, biais possible |
| Imputer | on remplace par une valeur | fausse précision |
| **Conserver et signaler** | on ne touche à rien, on rapporte les effectifs | analyse plus lourde |

Voyons ce que coûte réellement l'imputation.

In [ ]:
salaries = df.loc[df["STATUT"] != 3].copy()
salaries["impute"] = salaries["SALAIRE_NET"].fillna(salaries["SALAIRE_NET"].mean())

comparaison = pd.DataFrame({
    "conserver": [
        salaries["SALAIRE_NET"].mean(),
        salaries["SALAIRE_NET"].std(),
        salaries["SALAIRE_NET"].corr(salaries["DIPLOME"]),
        salaries["SALAIRE_NET"].notna().sum(),
    ],
    "imputer": [
        salaries["impute"].mean(),
        salaries["impute"].std(),
        salaries["impute"].corr(salaries["DIPLOME"]),
        salaries["impute"].notna().sum(),
    ],
}, index=["moyenne", "ecart-type", "correlation avec DIPLOME", "observations"])

comparaison.round(3)

**La moyenne ne bouge pas d'un centime.** C'est précisément ce qui rend l'opération dangereuse : rien ne signale que quelque chose a changé.

Ce qui change, c'est l'écart-type, la corrélation avec le diplôme, et le nombre d'observations que vous allez annoncer. Les erreurs-types calculées ensuite seront trop petites, et l'inférence trop confiante.

**Imputer, c'est faire une hypothèse sur ce qu'on ne voit pas.** C'est un modèle, pas une réparation, et un modèle se déclare.

## 13. Quand l'absence est une information

`PRIME` vide ne veut pas dire « non renseigné ». Le dictionnaire dit : **aucune prime perçue**.

In [ ]:
# Les deux vides doivent etre separes AVANT tout calcul : on repart du brut.
sal = brut.drop_duplicates()
sal = sal.loc[sal["STATUT"] != 3]

nr_prime = sal["PRIME"] == 999999          # non-reponse : on ne sait pas
connu    = sal.loc[~nr_prime, "PRIME"]     # statut de prime connu

print("salaries au statut de prime connu :", len(connu))
print("dont primes reellement percues    :", connu.notna().sum())
print()
print("moyenne des primes versees :", round(connu.mean(), 2))
print("prime moyenne par salarie  :", round(connu.fillna(0).mean(), 2))

Les deux chiffres sont justes. Ils ne répondent pas à la même question, et l'écart est d'un facteur deux.

Le premier décrit les salariés qui touchent une prime. Le second décrit l'ensemble des salariés dont on connaît le statut de prime. **Dites lequel vous avez calculé.**

Notez ce que ce calcul a exigé : repartir du fichier brut. Une fois `nettoyer()` passé, les deux sortes de vide sont confondues dans le même `NaN`, et l'information est perdue.

---

## 14. Le journal de nettoyage

Le livrable de cette séance n'est pas un fichier propre. C'est la trace des décisions, avec l'effet chiffré de chacune.

In [ ]:
journal = []


def noter(etape, decision, data, justification):
    journal.append({
        "etape": etape,
        "decision": decision,
        "lignes": len(data),
        "salaire_moyen": round(data["SALAIRE_NET"].mean(), 2),
        "obs_salaire": int(data["SALAIRE_NET"].notna().sum()),
        "justification": justification,
    })
    return pd.DataFrame(journal)


w = brut.copy()
noter("Chargement", "---", w, "Fichier brut")

w = w.drop_duplicates()
noter("Doublons techniques", "Suppression", w, "Lignes identiques partout")

w["SALAIRE_NET"] = w["SALAIRE_NET"].replace(999999, np.nan)
noter("Code 999999 -> manquant", "Conversion", w, "Dictionnaire, SALAIRE_NET")

w = w.loc[w["STATUT"] != 3]
noter("Non-salaries", "Exclusion", w, "Hors champ : pas de salaire")

Trois observations sur ce tableau.

- Supprimer douze lignes sur huit mille ne change presque rien.
- Traiter un seul code de non-réponse fait passer la moyenne de 67 745 à 2 375. **Une ligne de code, un facteur vingt-huit.**
- Retirer les non-salariés la fait remonter à 2 610, parce que leurs zéros tiraient la moyenne vers le bas.

Ce tableau tient en cinq lignes de notebook et rend votre travail vérifiable. Il constitue aussi, presque tel quel, la note méthodologique d'un mémoire de M1.

## 15. Le tableau que vous publierez

In [ ]:
sup = salaries["DIPLOME"] >= 5      # bac+2 et au-dela ; DIPLOME 9 est deja NaN

descriptif = pd.DataFrame({
    "moyenne": [
        salaries["SALAIRE_NET"].mean(),
        salaries["AGE"].mean(),
        salaries["HEURES"].mean(),
        sup.where(salaries["DIPLOME"].notna()).mean(),
    ],
    "observations": [
        salaries["SALAIRE_NET"].notna().sum(),
        salaries["AGE"].notna().sum(),
        salaries["HEURES"].notna().sum(),
        salaries["DIPLOME"].notna().sum(),
    ],
}, index=["Salaire mensuel net", "Age", "Heures hebdomadaires",
          "Part de diplomes du superieur"])

descriptif.round(2)

Chaque ligne repose sur un nombre différent d'observations, et le tableau ne dit pas pourquoi.

Trois questions auxquelles il ne répond pas :

1. Où sont passées les 686 lignes qui manquent par rapport au fichier initial ?
2. Les salaires nuls ont-ils été conservés ?
3. Le plafond de 10 000 euros est-il signalé quelque part ?

Toutes les réponses sont dans la note méthodologique — si elle existe.

---

## À faire avant la séance 4

1. Terminez votre journal de nettoyage : ajoutez-y une ligne par décision que vous prenez.
2. Recalculez le salaire moyen après chaque étape et notez l'écart.
3. Reprenez `CONTRAT` et `TEMPS` dans le dictionnaire. Quels codes sont des non-réponses, lesquels sont du hors champ ? Traitez-les.
4. Refaites la comparaison de la section 12 en imputant par la **médiane** au lieu de la moyenne. La conclusion change-t-elle ?
5. Vérifiez que le notebook s'exécute de haut en bas.

`stefaniamarcassa.github.io/analyse_des_donnees`